In [ ]:
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax
from torch.utils.data import DataLoader, Dataset
import copy
import math
import random
import warnings

warnings.filterwarnings("ignore")

# ==================== 使用模拟数据（不依赖网络下载）====================

class SyntheticTranslationDataset(Dataset):
    """生成模拟的翻译数据集用于测试"""
    def __init__(self, num_samples=5000, src_vocab_size=1000, tgt_vocab_size=1000, max_len=20):
        self.num_samples = num_samples
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.max_len = max_len
        
        # 生成模拟数据
        self.data = []
        for _ in range(num_samples):
            # 随机生成长度
            src_len = random.randint(5, max_len)
            tgt_len = random.randint(5, max_len)
            
            # 生成随机 token 序列（模拟句子）
            # 使用 1 到 vocab_size-1 之间的数字，0 留给 padding
            src_seq = [random.randint(1, src_vocab_size-1) for _ in range(src_len)]
            tgt_seq = [random.randint(1, tgt_vocab_size-1) for _ in range(tgt_len)]
            
            self.data.append((src_seq, tgt_seq))
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.data[idx]


# ==================== Transformer 模型组件 ====================

def clones(module, N):
    """产生 N 个相同的层"""
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])


class LayerNorm(nn.Module):
    """层归一化"""
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2


class SublayerConnection(nn.Module):
    """残差连接 + 层归一化"""
    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))
    #残差连接核心代码，对返回值的限定


class EncoderLayer(nn.Module):
    """编码器层"""
    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)
#包括自注意力层与前馈神经网络
#每层内有层归一化，Dropout与残差连接

class Encoder(nn.Module):
    """编码器"""
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)
    #堆叠多个编码器层，统一归一化


class DecoderLayer(nn.Module):
    """解码器层"""
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)


class Decoder(nn.Module):
    """解码器"""
    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


def attention(query, key, value, mask=None, dropout=None):
    """缩放点积注意力"""
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn


class MultiHeadedAttention(nn.Module):
    """多头注意力"""
    def __init__(self, h, d_model, dropout=0.1):
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]
        
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)
        
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        return self.linears[-1](x)


class PositionwiseFeedForward(nn.Module):
    """位置前馈网络"""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))


class PositionalEncoding(nn.Module):
    """位置编码"""
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)].requires_grad_(False)
        return self.dropout(x)


class EncoderDecoder(nn.Module):
    """完整的编码器-解码器模型"""
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator
        
    def forward(self, src, tgt, src_mask, tgt_mask):
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)
    
    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)
    
    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)


class Embeddings(nn.Module):
    """词嵌入层"""
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model
        
    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)
    #将整数索引替换为对应向量，缩放保持尺度


class Generator(nn.Module):
    """生成输出层"""
    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab)
        
    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)


def make_model(src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1):
    """构建完整的 Transformer 模型"""
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab)
    )
    
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model


def subsequent_mask(size):
    """生成后续掩码"""
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(torch.uint8)
    return subsequent_mask == 0


def collate_batch(batch, pad_idx=0):
    """批处理函数"""
    src_batch, tgt_batch = [], []
    for src, tgt in batch:
        src_batch.append(torch.tensor(src, dtype=torch.long))
        tgt_batch.append(torch.tensor(tgt, dtype=torch.long))
    
    src_batch = nn.utils.rnn.pad_sequence(src_batch, padding_value=pad_idx, batch_first=True)
    tgt_batch = nn.utils.rnn.pad_sequence(tgt_batch, padding_value=pad_idx, batch_first=True)
    
    return src_batch, tgt_batch


class Batch:
    """批数据封装"""
    def __init__(self, src, tgt=None, pad=0):
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2)
        if tgt is not None:
            self.tgt = tgt[:, :-1]
            self.tgt_y = tgt[:, 1:]
            self.tgt_mask = self.make_std_mask(self.tgt, pad)
            self.ntokens = (self.tgt_y != pad).data.sum()
    
    @staticmethod
    def make_std_mask(tgt, pad):
        tgt_mask = (tgt != pad).unsqueeze(-2)
        tgt_mask = tgt_mask & subsequent_mask(tgt.size(-1)).type_as(tgt_mask.data)
        return tgt_mask


class SimpleLossCompute:
    """简单损失计算"""
    def __init__(self, generator, criterion):
        self.generator = generator
        self.criterion = criterion
    
    def __call__(self, x, y, norm):
        x = self.generator(x)
        loss = self.criterion(x.contiguous().view(-1, x.size(-1)), y.contiguous().view(-1))
        return loss / norm


def rate(step, model_size, factor, warmup):
    """学习率调度函数"""
    if step == 0:
        step = 1
    return factor * (model_size ** (-0.5) * min(step ** (-0.5), step * warmup ** (-1.5)))


# ==================== 主程序 ====================

def main():
    # 检查 GPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    # 参数设置
    src_vocab_size = 1000  # 源语言词表大小
    tgt_vocab_size = 1000  # 目标语言词表大小
    pad_idx = 0  # padding 的索引
    
    # 生成模拟数据
    print("="*50)
    print("生成模拟训练数据...")
    print("="*50)
    
    train_dataset = SyntheticTranslationDataset(
        num_samples=5000, 
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        max_len=30
    )
    
    valid_dataset = SyntheticTranslationDataset(
        num_samples=500,
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        max_len=30
    )
    
    print(f"训练集大小: {len(train_dataset)}")
    print(f"验证集大小: {len(valid_dataset)}")
    
    # 创建 DataLoader
    batch_size = 32
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True,
        collate_fn=lambda batch: collate_batch(batch, pad_idx)
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda batch: collate_batch(batch, pad_idx)
    )
    
    # 创建模型
    print("\n" + "="*50)
    print("构建 Transformer 模型...")
    print("="*50)
    
    model = make_model(
        src_vocab_size, 
        tgt_vocab_size, 
        N=3,          # 编码器/解码器层数
        d_model=128,  # 模型维度（减小以加速训练）
        d_ff=256,     # 前馈网络维度
        h=4,          # 注意力头数
        dropout=0.1
    )
    model.to(device)
    
    # 计算模型参数量
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"总参数量: {total_params:,}")
    print(f"可训练参数量: {trainable_params:,}")
    
    # 优化器和损失函数
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
    criterion = nn.NLLLoss(ignore_index=pad_idx)
    loss_compute = SimpleLossCompute(model.generator, criterion)
    
    # 学习率调度器
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, 
        lr_lambda=lambda step: rate(step, 128, 1, 4000)
    )
    
    # 训练循环
    num_epochs = 5
    print("\n" + "="*50)
    print(f"开始训练 {num_epochs} 个 epoch")
    print("="*50)
    
    global_step = 0
    
    for epoch in range(num_epochs):
        # 训练
        model.train()
        train_loss = 0
        train_steps = 0
        
        for batch_idx, (src, tgt) in enumerate(train_loader):
            src = src.to(device)
            tgt = tgt.to(device)
            batch = Batch(src, tgt, pad_idx)
            
            out = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
            loss = loss_compute(out, batch.tgt_y, batch.ntokens)
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
            
            train_loss += loss.item()
            train_steps += 1
            global_step += 1
            
            if batch_idx % 50 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}, Loss: {loss.item():.4f}")
        
        avg_train_loss = train_loss / train_steps
        
        # 验证
        model.eval()
        valid_loss = 0
        valid_steps = 0
        
        with torch.no_grad():
            for src, tgt in valid_loader:
                src = src.to(device)
                tgt = tgt.to(device)
                batch = Batch(src, tgt, pad_idx)
                out = model(batch.src, batch.tgt, batch.src_mask, batch.tgt_mask)
                loss = loss_compute(out, batch.tgt_y, batch.ntokens)
                valid_loss += loss.item()
                valid_steps += 1
        
        avg_valid_loss = valid_loss / valid_steps
        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Valid Loss = {avg_valid_loss:.4f}")
    
    print("\n" + "="*50)
    print("训练完成！")
    print("="*50)
    
    # 保存模型
    torch.save(model.state_dict(), 'transformer_model.pt')
    print("模型已保存为 transformer_model.pt")
    
    return model


if __name__ == "__main__":
    model = main()
    
    print("\n" + "="*50)
    print("="*50)

使用设备: cuda
生成模拟训练数据...
训练集大小: 5000
验证集大小: 500

构建 Transformer 模型...
总参数量: 1,379,304
可训练参数量: 1,379,304

开始训练 5 个 epoch
Epoch 1/5, Batch 0, Loss: 0.0126
Epoch 1/5, Batch 50, Loss: 0.0121
Epoch 1/5, Batch 100, Loss: 0.0118
Epoch 1/5, Batch 150, Loss: 0.0116
Epoch 1: Train Loss = 0.0137, Valid Loss = 0.0138
Epoch 2/5, Batch 0, Loss: 0.0144
Epoch 2/5, Batch 50, Loss: 0.0134
Epoch 2/5, Batch 100, Loss: 0.0127
Epoch 2/5, Batch 150, Loss: 0.0135
Epoch 2: Train Loss = 0.0137, Valid Loss = 0.0138
Epoch 3/5, Batch 0, Loss: 0.0140
Epoch 3/5, Batch 50, Loss: 0.0123
Epoch 3/5, Batch 100, Loss: 0.0133
Epoch 3/5, Batch 150, Loss: 0.0131
Epoch 3: Train Loss = 0.0136, Valid Loss = 0.0138
Epoch 4/5, Batch 0, Loss: 0.0137
Epoch 4/5, Batch 50, Loss: 0.0138
Epoch 4/5, Batch 100, Loss: 0.0152
Epoch 4/5, Batch 150, Loss: 0.0136
Epoch 4: Train Loss = 0.0136, Valid Loss = 0.0138
Epoch 5/5, Batch 0, Loss: 0.0114
Epoch 5/5, Batch 50, Loss: 0.0117
Epoch 5/5, Batch 100, Loss: 0.0127
Epoch 5/5, Batch 150, Loss: 0.01

In [ ]:
def subsequent_mask(size):
    """生成后续掩码（上三角矩阵）"""
    # 创建上三角矩阵（对角线以上为1）
    mask = torch.triu(torch.ones(1, size, size), diagonal=1)
    # 取反（变成0表示要遮住的位置）
    return mask == 0

# 例子
size = 4
mask = subsequent_mask(4)
print(mask)
# tensor([[[True, False, False, False],
#          [True, True, False, False],
#          [True, True, True, False],
#          [True, True, True, True]]])


class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super().__init__()
        self.h = h
        self.d_k = d_model // h
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)
        
        # 线性投影层
        self.linears = clones(nn.Linear(d_model, d_model), 4)
    
    def forward(self, query, key, value, mask=None):
        if mask is not None:
            # 掩码形状: [batch, 1, 1, seq_len] -> [batch, h, seq_len, seq_len]
            mask = mask.unsqueeze(1)
        
        # 1. 投影到多个头
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]
        
        #计算注意力（应用掩码）
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)
        
        # 4. 输出投影
        return self.linears[-1](x)


def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        # 将掩码为0的位置设为负无穷
        scores = scores.masked_fill(mask == 0, -1e9)
    
    p_attn = scores.softmax(dim=-1)
    
    if dropout is not None:
        p_attn = dropout(p_attn)
    
    return torch.matmul(p_attn, value), p_attn

In [ ]:
class MultiHeadCrossAttention(nn.Module):
    """多头交叉注意力"""
    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0
        
        self.d_k = d_model // n_head
        self.n_head = n_head
        
        # Q, K, V 的投影层
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, decoder_out, encoder_out, mask=None):
        """
        decoder_out: 解码器的输出 [batch, tgt_len, d_model]
        encoder_out: 编码器的输出 [batch, src_len, d_model]
        mask: 可选的掩码（如 padding mask）
        """
        batch_size, tgt_len, _ = decoder_out.shape
        src_len = encoder_out.shape[1]
        
        # 1. 线性投影
        # Q 来自解码器
        Q = self.w_q(decoder_out).view(batch_size, tgt_len, self.n_head, self.d_k)
        # K, V 来自编码器
        K = self.w_k(encoder_out).view(batch_size, src_len, self.n_head, self.d_k)
        V = self.w_v(encoder_out).view(batch_size, src_len, self.n_head, self.d_k)
        
        # 2. 转置以便计算 [batch, head, seq, d_k]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        # 3. 计算注意力分数
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # 4. 应用掩码（如果需要）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # 5. Softmax 和 dropout
        attn_weights = scores.softmax(dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # 6. 加权求和
        attn_output = torch.matmul(attn_weights, V)
        
        # 7. 合并多头
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, tgt_len, -1)
        
        # 8. 输出投影
        return self.w_o(attn_output)